# Creating and Managing Adapters and Pipelines with Python

The previous tutorials used resources that were already available in StreamPipes. In this tutorial, we will create an adapter and a complete pipeline directly from Python.

Our example monitors simulated flow-rate measurements. A Boolean Filter keeps events for which `sensor_fault_flags` is `False`, and a Data Lake sink stores these healthy measurements. We will start the resources, query the Data Lake to verify that events arrive, and finally remove the adapter and pipeline again.

## What we are going to build

The tutorial creates the following data flow:

```text
Machine Data Simulator → Boolean Filter → Data Lake
```

We will perform the following steps:

1. Connect the Python client to StreamPipes.
2. Create a Machine Data Simulator adapter with a compact Python model.
3. Create a Boolean Filter and Data Lake pipeline from YAML.
4. Start the pipeline and adapter.
5. Query the Data Lake and inspect the stored events.
6. Stop and delete the created processing resources.

This example assumes that the Machine Data Simulator, Boolean Filter, and Data Lake sink are available in your StreamPipes installation.

## Connecting to your StreamPipes instance

As in the previous tutorials, we first create a client and connect it to our StreamPipes instance. If you are not yet familiar with the client configuration, have a look at our [first tutorial](../1-introduction-to-streampipes-python-client).

In [ ]:
import os
os.environ["SP_USERNAME"] = "admin@streampipes.apache.org"
os.environ["SP_API_KEY"] = "INSERT-API-KEY"

In [ ]:
from time import monotonic, sleep
from uuid import uuid4

from streampipes.client import StreamPipesClient
from streampipes.client.config import StreamPipesClientConfig
from streampipes.client.credential_provider import StreamPipesApiKeyCredentials
from streampipes.model.compact import CompactAdapter, CompactEventProperty, CompactPipeline, CreateOptions

config = StreamPipesClientConfig(
    credential_provider=StreamPipesApiKeyCredentials(),
    host_address="localhost",
    port=80,
    https_disabled="true",
)
client = StreamPipesClient(config)
client.describe()

## Creating the simulator adapter

Adapters connect StreamPipes to external data sources. Here, we use the built-in Machine Data Simulator and configure it to emit one flow-rate event every 500 milliseconds.

The `CompactAdapter` model contains the adapter name, the application identifier of the adapter extension, and its configuration. We set both creation options to `False` because we want to start the adapter ourselves and create a custom persistence pipeline in the next section.

In [ ]:
adapter_name = "Python tutorial simulator"
adapter_id = str(uuid4())

adapter = CompactAdapter(
    id=adapter_id,
    name=adapter_name,
    description="Flow-rate simulator created by the Python client tutorial",
    app_id="org.apache.streampipes.connect.iiot.adapters.simulator.machine",
    configuration=[
        {"wait-time-ms": "500"},
        {"numberOfSensors": "1"},
        {"selected-simulator-option": "flowrate"},
    ],
    schema={
        "mass_flow": CompactEventProperty(
            property_scope="MEASUREMENT_PROPERTY",
        ),
        "density": CompactEventProperty(
            property_scope="MEASUREMENT_PROPERTY",
        ),
        "volume_flow": CompactEventProperty(
            property_scope="MEASUREMENT_PROPERTY",
        ),
        "sensor_fault_flags": CompactEventProperty(
            property_scope="MEASUREMENT_PROPERTY",
        ),
        "temperature": CompactEventProperty(
            property_scope="MEASUREMENT_PROPERTY",
        ),
        "sensorId": CompactEventProperty(
            property_scope="MEASUREMENT_PROPERTY",
        ),
        "timestamp": CompactEventProperty(
            property_scope="HEADER_PROPERTY",
            semantic_type="http://schema.org/DateTime",
            additional_metadata={
                "originType": "http://www.w3.org/2001/XMLSchema#float",
            },
        ),
    },
    create_options=CreateOptions(persist=False, start=False),
)


client.adapterApi.post(adapter)


The creation request does not return the adapter itself. We therefore retrieve the adapter summary. The summary contains the data stream identifier required by the pipeline.

In [ ]:
adapter = client.adapterApi.get(adapter_id)
stream_id = adapter.corresponding_data_stream_element_id
adapter


## Creating the pipeline from YAML

Compact resources can be created with Python models or loaded from JSON and YAML. For the pipeline, we use YAML so that the complete data flow is easy to recognize and copy.

Every pipeline element has a `ref`. The `connectedTo` entries use these references to form the pipeline graph:

- `simulator` refers to the stream created by our adapter.
- `healthy-events` maps `sensor_fault_flags` and only forwards events whose value is `False`.
- `data-lake` stores the filtered events under a unique measurement name.

The Data Lake sink uses the event's `timestamp` field as its time index and `sensorId` as a dimension.

In [ ]:
pipeline_name = "Python tutorial pipeline"
measurement_name = "python-tutorial-pipeline"
pipeline_id = str(uuid4())

pipeline_yaml = f"""
id: {pipeline_id}
name: {pipeline_name}
description: Store healthy flow-rate measurements in the Data Lake
pipelineElements:
  - type: stream
    ref: simulator
    id: {stream_id}

  - type: processor
    ref: healthy-events
    id: org.apache.streampipes.processors.filters.jvm.processor.booleanfilter
    connectedTo:
      - simulator
    configuration:
      - boolean-mapping: "s0::sensor_fault_flags"
      - value: "False"

  - type: sink
    ref: data-lake
    id: org.apache.streampipes.sinks.internal.jvm.datalake
    connectedTo:
      - healthy-events
    configuration:
      - timestamp_mapping: "s0::timestamp"
      - db_measurement: {measurement_name}
      - schema_update: Update schema
      - dimensions_selection:
          - sensorId
      - ignore_duplicates: false

createOptions:
  start: false
"""


`CompactPipeline.from_yaml()` parses and validates the YAML representation. We can then submit the resulting model just like a model constructed directly in Python.

In [ ]:
pipeline = CompactPipeline.from_yaml(pipeline_yaml)
client.pipelineApi.post(pipeline)

client.pipelineApi.all().to_pandas()

## Starting the pipeline and adapter

Both resources were created with `start=False`. We start the pipeline first so that it is ready to receive events, and then start the simulator adapter.

In [ ]:
client.pipelineApi.start(pipeline_id)
client.adapterApi.start(adapter_id)


## Verifying the stored data

Event processing and Data Lake writes happen asynchronously. The helper below waits until the measurement is available and contains at least one event. It then returns up to 100 of the newest events as a pandas `DataFrame`.

Since the Boolean Filter only keeps this value, every row returned by our query should contain `False` in that column.

In [ ]:
def wait_for_stored_events(measurement: str, timeout_seconds: float = 60.0):
    deadline = monotonic() + timeout_seconds

    while monotonic() < deadline:
        available_measurements = client.dataLakeMeasureApi.all()
        measurement_exists = any(
            item.measure_name == measurement for item in available_measurements
        )

        if measurement_exists:
            result = client.dataLakeMeasureApi.get(
                identifier=measurement,
                limit=100,
                order="DESC",
            )
            if result.total > 0:
                return result.to_pandas()

        sleep(2)

    raise TimeoutError(f"No events were stored in {measurement!r} within the timeout")


stored_events = wait_for_stored_events(measurement_name)
stored_events


We can inspect the values selected by the Boolean Filter. The result should only contain `False`.

In [ ]:
stored_events["sensor_fault_flags"].value_counts()


## Cleaning up

We stop the simulator first so that no new events enter the pipeline. We then stop and delete the pipeline before deleting the adapter on which it depends.

Deleting these processing resources does not delete historical measurements. The measurement created by this tutorial remains available in the Data Lake. You can inspect or delete it later in StreamPipes.

In [ ]:
client.adapterApi.stop(adapter_id)
client.pipelineApi.stop(pipeline_id)

client.pipelineApi.delete(pipeline_id)
client.adapterApi.delete(adapter_id)


How do you like this tutorial?
We hope you like it and would love to receive some feedback from you.
Just go to our [GitHub discussion page](https://github.com/apache/streampipes/discussions) and let us know your impression.
We'll read and react to them all, we promise!